## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-18-21-55-10 +0000,nypost,Major warning for electric car owners in Calif...,https://nypost.com/2026/11/18/us-news/californ...
1,2026-11-18-21-54-02 +0000,nypost,"Gavin Newsom releases rogue AI report, calls f...",https://nypost.com/2026/11/18/us-news/gavin-ne...
2,2026-11-18-21-51-00 +0000,wsj,U.S. Natural Gas Futures Post Strong Gains,https://www.wsj.com/finance/commodities-future...
3,2026-11-18-21-46-34 +0000,chitri,US Sen. Tammy Duckworth says she won’t write ‘...,https://www.chicagotribune.com/2026/11/18/tamm...
4,2026-11-18-21-46-00 +0000,wsj,Opinion | The Shakespearean Tragedy of King Do...,https://www.wsj.com/opinion/the-shakespearean-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2659/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
127,trump,59
211,nvidia,28
180,new,21
97,tech,19
581,xi,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
428,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...,157
132,2026-11-18-18-41-56 +0000,chitri,President Trump plans separate talks with Vlad...,https://www.chicagotribune.com/2026/11/18/trum...,152
18,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...,151
130,2026-11-18-18-52-04 +0000,nyt,Trump Arrives in China Focused on Friendship W...,https://www.nytimes.com/2026/11/18/us/politics...,138
259,2026-11-18-12-07-00 +0000,wsj,"In Choosing Shenzhen for Trump-Putin Summit, X...",https://www.wsj.com/world/china/in-choosing-sh...,135


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
428,157,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...
356,102,2026-11-18-04-23-00 +0000,wsj,Democrat Mary Peltola unseated Republican Sen....,https://www.wsj.com/politics/elections/democra...
18,88,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...
47,76,2026-11-18-21-04-35 +0000,nyt,Jensen Huang Is Top of Mind for Tech Leaders a...,https://www.nytimes.com/2026/11/18/business/de...
16,63,2026-11-18-21-40-00 +0000,nypost,LA mayor-elect Nithya Raman skips LA City Coun...,https://nypost.com/2026/11/18/us-news/la-mayor...
304,53,2026-11-18-10-03-56 +0000,nyt,Putin Will Continue Ukraine War Despite Plans ...,https://www.nytimes.com/2026/11/18/us/politics...
377,51,2026-11-18-02-14-16 +0000,latimes,Franklin High teen instigated physical alterca...,https://www.latimes.com/california/story/2026-...
177,51,2026-11-18-17-03-55 +0000,cbc,Hakeem Jeffries poised to become 1st Black U.S...,https://www.cbc.ca/news/world/us-democrats-hou...
264,47,2026-11-18-11-57-00 +0000,wsj,Homeland Security Secretary Markwayne Mullin h...,https://www.wsj.com/politics/policy/markwayne-...
121,41,2026-11-18-19-19-46 +0000,nypost,Nasty nabe: Two women accused of separate sick...,https://nypost.com/2026/11/18/us-news/perv-tea...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
